# Chat and Instruct Models



## References


* [Hugging Face Chat Basics](https://huggingface.co/docs/transformers/en/conversations)
* [SmolLM2: When Smol Goes Big — Data-Centric Training of a Small Language Model](https://arxiv.org/pdf/2502.02737)


## Install Modules

We'll be using `transformers` version 5. 


In [2]:
import sys
!{sys.executable} -m pip install transformers accelerate

/bin/bash: /Users/sagar/School/Drake: No such file or directory


## Large vs. Small language models

Large Language Models (GPT 5.2, Claude Opus 4.6, Grok 4.1, Gemini 3) get all the attention.

Smaller language models have come a long way too, and they require much less computation
* can often be run on a laptop or a Colab instance
* can be *fine-tuned* for specific applications with good performance


### Example: SmolLM2

Hugging Face developed a [family of small language models called SmolLM](https://huggingface.co/collections/HuggingFaceTB/smollm2) there's also a [SmolLM3](https://huggingface.co/blog/smollm3)

SmolLM2 comes in various sizes
* 135M (135 million parameters - weights in the neural network)
* 360M
* 1.7B

Contrast with the LLMs above which likely all have over 100 billion parameters and run on a cluster of devices in a data center

Each size has a **base model**, like [SmolLM2-360M](https://huggingface.co/HuggingFaceTB/SmolLM2-360M)
* *pre-trained* on lots of diverse text
* designed to predict the *next word* - it's your phone's keyboard text prediction on steroids

The *base model* is then fine-tuned on *instruction following* and *conversational data*, which make it useful for building **chat bots**.
* resulting model has a name like [SmolLM2-360M-Instruct](https://huggingface.co/HuggingFaceTB/SmolLM2-360M-Instruct)

## Building a chat bot with the `text-generation` pipeline

Setting up a chat bot works the same way as other Hugging Face models we've seen, but we'll use the `text-generation` pipeline

In [3]:
from transformers import pipeline
from accelerate import Accelerator

device = Accelerator().device

chatbot = pipeline("text-generation", model="HuggingFaceTB/SmolLM2-360M-Instruct", device = device)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

### Chat template

*Instruct* models often allow you to pass the input in using a **Chat Template** like this

In [6]:
chat_history = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Explain gravity in one paragraph."},
]

Now lets get the response and display what is returned

In [7]:
response = chatbot(chat_history)
response

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': [{'role': 'system',
    'content': 'You are a helpful assistant.'},
   {'role': 'user', 'content': 'Explain gravity in one paragraph.'},
   {'role': 'assistant',
    'content': 'Gravity is a fundamental force of nature that draws objects toward each other, making it the most powerful and ubiquitous force in the universe. It occurs due to the mutual attraction between masses, such as planets, stars, and objects. This attraction is typically described by the universal law of gravity, which states that every point mass attracts every other point mass by a force acting along the line intersecting both points and proportional to the product of their masses and inversely proportional to the square of the distance between them.'}]}]

**Notice:**
* it just uses normal lists and dictionaries
* it returns the entire chat history on the `'generated_text'` key

If you wanted to just print out the response, you could do it like this

In [8]:
print("ASSISTANT:", response[0]['generated_text'][-1]['content'])

ASSISTANT: Gravity is a fundamental force of nature that draws objects toward each other, making it the most powerful and ubiquitous force in the universe. It occurs due to the mutual attraction between masses, such as planets, stars, and objects. This attraction is typically described by the universal law of gravity, which states that every point mass attracts every other point mass by a force acting along the line intersecting both points and proportional to the product of their masses and inversely proportional to the square of the distance between them.


## Multi-turn conversations

If you want to have a multi-turn conversation, you need to pass *the entire chat history* to the model - it doesn't have any inherent memory of the output it just gave you

In [9]:
another_chat_history = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Explain gravity in one paragraph."},
    {"role": "assistant", "content": "Gravity is a fundamental force of nature that causes objects with mass to attract each other. According to Sir Isaac Newton, the force of gravity between two objects depends on their masses and the distance between them. The larger the mass of the objects, the greater the gravitational pull. This force is responsible for keeping planets in orbit around the sun and causes objects to fall towards the ground when dropped. Albert Einstein also described gravity as a curvature of spacetime caused by massive objects, which in turn warps the fabric of spacetime around them."},
    {"role": "user", "content": "Which of those two do you think has had a bigger impact on the field?"}
]

next_response = chatbot(another_chat_history)
print(next_response[0]['generated_text'][-1]['content'])

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


I would say that Einstein's theory of general relativity has had a much bigger impact on the field of astrophysics and cosmology. General relativity provides a comprehensive description of gravity and the behavior of gravity, which is essential for understanding phenomena such as black holes, gravitational waves, and the expansion of the universe. It has also led to numerous breakthroughs in our understanding of the universe, including the detection of gravitational waves by the Laser Interferometer Gravitational-Wave Observatory (LIGO).


In [11]:
#writing a loop that allows for back-and-forth conversation with the model to keep track of full history of the chat as we go

messages = another_chat_history.copy()

print("Type 'quit' to stop")

while True:
   
    user_input = input("You: ")
    
    
    if user_input.lower() in ["quit", "exit", "stop"]: # i need something to break this loop 
        print("Chatbot: Nice to talk to you!")
        break
    
    messages.append({"role": "user", "content": user_input})

   
    response_data = chatbot(messages, max_new_tokens=50, continuation_token=None) # grab that response 
    
    
    assistant_message = response_data[0]['generated_text'][-1]['content']
    
    
    print(f"Assistant: {assistant_message}")
    messages.append({"role": "assistant", "content": assistant_message})

Type 'quit' to stop


Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Assistant: Both concepts of gravity have had a significant impact on the field of physics. Newton's law of universal gravitation lays the foundation for understanding gravity and its role in the behavior of celestial bodies. Einstein's theory of general relativity expanded our understanding of gravity beyond


Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Assistant: Acknowledging the impact of both ideas on physics, Einstein's theory of general relativity revolutionized our understanding of gravity and the universe's structure. His work, in particular, led to the development of modern astrophysics, including the concept of black holes


...

Assistant: You're a student at Drake University. We're happy to have you here! What can we help you with today?


Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Assistant: You're likely one of our students at Drake University. Don't hesitate to ask us anything about our services or product offerings.


Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Assistant: You're likely going on a field trip to the sports facility. We're happy to help with any questions or concerns you might have.
Chatbot: Nice to talk to you!


In [13]:
messages = another_chat_history.copy()

print("Type exit to stop")

while True:
   
    user_input = input("You: ").strip()
    if user_input.lower() in ["exit", "quit", "stop"]:
        break
    
    messages.append({"role": "user", "content": user_input})

    
    output = chatbot(messages, max_new_tokens=10000)

   
    new_assistant_content = output[0]['generated_text'][-1]['content']
    
    
    print(f"Assistant: {new_assistant_content}")
    messages.append({"role": "assistant", "content": new_assistant_content})

Type exit to stop


Both `max_new_tokens` (=10000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Assistant: What?


Both `max_new_tokens` (=10000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Assistant: I'm sorry for the confusion, but I'm here to assist with mathematical and programming-related queries. I can help with solving math problems, explaining mathematical concepts, and providing explanations for various programming languages. I'm not equipped to answer questions about other topics.


...

Assistant: Hey Sagar, congratulations on your new position as a Junior Researcher! That's an amazing honor.

I was thinking, since you're interested in Python, you might find the color blue appealing. It's a color that's calming and soothing, which can be very beneficial for people who work with data. Blue is also a color that can help to reduce stress and anxiety, making it a great choice for a new role.

You might also like the color green, which is often associated with growth, sustainability, and nature. Green is a great color to represent the environment and eco-friendly initiatives, which could be a plus for your work.

What do you think? Do you have any favorite colors or colors that evoke a particular mood or feeling?


Both `max_new_tokens` (=10000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Assistant: Great to hear that you like blue! It's a beautiful color that can be very calming and soothing. You might also like the color green, which can represent growth, sustainability, and nature.

Blue is often associated with stability, trustworthiness, and wisdom, which can be very beneficial for a new role. It's a color that can help to create a sense of calm and focus, which can be very beneficial for people who work with data.

I think you might also like the color green, which can represent growth, sustainability, and nature. Green is often associated with eco-friendly initiatives and environmental protection, which can be a plus for your work.

What do you like most about these colors? Do you have any favorite brands, companies, or organizations that use these colors?


Both `max_new_tokens` (=10000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Assistant: Hello, I'm glad to meet you. My name is Alex Chen, and I'm a senior systems engineer at the Drake Engineering Conference team.

I'm excited to meet you and learn about your interests and goals. I've been working with Python for a few years now, and I think it's a great language to learn if you're interested in data science and machine learning.

What are your interests and goals? Are you interested in learning more about Python, or would you like to learn more about the types of projects I work on at the conference?


In [19]:
print("Type exit to quit.")

while True:
    user_input = input("You: ") 
    if user_input.lower() in ['exit', 'quit']:
        break
        
    chat_history.append({"role": "user", "content": user_input})
    
    response = chatbot(chat_history)
    
    assistant_message = response[0]['generated_text'][-1]['content']
    
    print(f"User:{user_input}")
    print(f"Assistant: {assistant_message}\n")
    
    chat_history.append({"role": "assistant", "content": assistant_message})

Type exit to quit.


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


User:i am sagar
Assistant: Sagar, nice to meet you. What's your favorite thing about Drake?



## Evaluating Chat Models: Benchmarks

A benchmark is a dataset with one or more reference answers that can be used to measure a model's response (like the reference summaries we compared against with ROUGE)


Model benchmarking is a big deal - companies like to report how well their models perform on all kinds of benchmarks

For example, see the **performance** tab here: https://deepmind.google/models/gemini/pro/

Take a look at this benchmark for multi-turn conversations: https://huggingface.co/datasets/HuggingFaceH4/mt_bench_prompts












* **Model 1**: [Qwen/Qwen2.5-0.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct) - I chose this because we looked at it in class. It's a 0.5 billion parameter instruct model from the Qwen team. It's known to be quite strong for its small size.
* **Model 2**: [HuggingFaceTB/SmolLM2-360M-Instruct](https://huggingface.co/HuggingFaceTB/SmolLM2-360M-Instruct) - This is the small 360M parameter model we used earlier in the lab. I wanted to see how it compares directly with Qwen2.5-0.5B-Instruct.

### My Fun Benchmark: Animal Trivia & Jokes!


**My 5 Prompts:**
1. "Tell me a funny joke about a penguin."
2. "What is the fastest land animal in the world, and what is its top speed?"
3. "Can you explain why flamingos are pink in exactly one sentence?"
4. "What is the collective noun for a group of crows?"
5. "If a bear and a shark played chess, who would win and why?"v

In [23]:
from transformers import pipeline
from accelerate import Accelerator

device = Accelerator().device

qwen_bot = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", device=device) #model 1
smol_bot = pipeline("text-generation", model="HuggingFaceTB/SmolLM2-360M-Instruct", device=device) #model 2

prompts = [
    "Tell me a funny joke about a penguin.",
    "What is the fastest land animal in the world, and what is its top speed?",
    "Can you explain why flamingos are pink in exactly one sentence?",
    "What is the collective noun for a group of crows?",
    "If a bear and a shark played chess, who would win and why?"
]

print(" Animal Trivia & Jokes Benchmark \n")

for i, prompt in enumerate(prompts):
    print(f"\nPrompt {i+1}: {prompt}")
    
    # Test Model1  Qwen
    chat_qwen = [{"role": "user", "content": prompt}]
    res_qwen = qwen_bot(chat_qwen, max_new_tokens=150)
    print(f"\n Qwen2.5-0.5B-Instruct says:\n{res_qwen[0]['generated_text'][-1]['content']}")
    
    # Test  SmolLM2
    chat_smol = [{"role": "user", "content": prompt}]
    res_smol = smol_bot(chat_smol, max_new_tokens=150)
    print(f"\n SmolLM2-360M-Instruct says:\n{res_smol[0]['generated_text'][-1]['content']}")
    
    print("-" * 80)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 Animal Trivia & Jokes Benchmark 


Prompt 1: Tell me a funny joke about a penguin.


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...


 Qwen2.5-0.5B-Instruct says:
The collective noun for a group of crows is "crowd". This term refers to a large group of birds that gather together in a particular area or location. Crows are known for their social behavior and often form complex groups with other members of their species. The term "crow" can also be used metaphorically to refer to a large group of people or animals, depending on context.


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



 SmolLM2-360M-Instruct says:
The collective noun for a group of crows is a "murder."
--------------------------------------------------------------------------------

Prompt 5: If a bear and a shark played chess, who would win and why?


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



 Qwen2.5-0.5B-Instruct says:
In chess, the game is traditionally between two players, with one player winning if they capture all opponent's pawns or pieces. However, in this hypothetical scenario where a bear and a shark play chess, we need to consider their characteristics and strategies.

Bear: Bears are large carnivores known for their strength, agility, and ability to hunt and kill prey. They have sharp claws, teeth, and can be highly maneuverable on land and water. Their size makes them formidable opponents, especially when playing against smaller creatures like sharks.

Shark: Sharks are also predators that rely heavily on speed, stealth, and strong muscles. They have streamlined bodies, powerful jaws, and can move swiftly through waters at high speeds. Their presence often creates chaos in the

 SmolLM2-360M-Instruct says:
If a bear and a shark played chess, it's unlikely that a shark would win. The reason for this is that the rules of chess are different from the rules of sha

### Benchmark Results

### Important : The answer is different for each time I try to generate response. Therefore, response above and my conclusion below might differ.

**1. Joke about a penguin**
I did not like none of their response to this question.

**2. Fastest land animal**
They both usually correctly identify the Cheetah (around 70-75 mph). They tie on factual retrieval for simple facts!

**3. Flamingos in one sentence**
Smaller models often struggle with length constraints. Often, SmolLm2 is slightly better at closely following the "exactly one sentence" instruction, whereas Qwen might accidentally produce two sentences or run-on sentences. They both usually correctly .

**4. Collective noun for crows**
Both got very incoorect.


**5. Chess: Bear vs. Shark**
This tests a bit of reasoning and humor. Because a shark can't breathe out of water and has no hands, the bear would win by default (or they couldn't play). It's fun to see which model hallucinates for this types of question . It was quite surprisindg for me that they both tried to presume or assume somehow , Human Chess Player which was not even part of asked question.

**Conclusion:**
While both models are surprisingly capable for under 1 billion parameters, **Qwen2.5-0.5B-Instruct** feels slightly better at strict instruction but they both did so so , not very satisfactory result.